In [3]:

"""
PMI (Perturbed Matsumoto-Imai) Kriptosistemi - Referans Uygulama
====================================================================

Bu modül, Mustafa SOLMAZ'ın "Kuantum Sonrası Çok Değişkenli Açık Anahtar
Kriptosistemlerinin İncelenmesi" başlıklı yüksek lisans tezinin 5.1.4.1
bölümünde ("PMI Kriptosistemi") teorik olarak sunulan yapının SageMath
ortamında somutlaştırılmış halidir.

Teorik Arka Plan
-----------------
Matsumoto-Imai (MI) kriptosistemi, Patarin tarafından geliştirilen
"lineerleştirme saldırısı" ile kırılmıştır. Bu saldırının temel dayanağı,
MI merkez dönüşümünün F(Y) = Y^(1+q^theta) biçiminde cebirsel olarak çok
katı bir yapıya sahip olması ve bu yapının açık anahtarda x ve şifreli
metin y arasında saldırgan tarafından bilinen bilineer (lineer) denklemler
üretmesidir.

PMI (Perturbed Matsumoto-Imai), bu zafiyeti gidermek amacıyla 2004 yılında
önerilmiştir. Temel fikir, MI merkez dönüşümünün çıktısına, küçük boyutlu
(r < n) bir "pertürbasyon" (gürültü) bileşeni eklemektir. Bu ek gürültü,
saldırganın kurmaya çalıştığı lineer ilişkileri bozarak doğrudan
lineerleştirme saldırısını etkisiz kılar. Buna karşılık, meşru alıcı;
gizli anahtar bilgisi sayesinde olası tüm pertürbasyon değerlerini (z)
tek tek deneyerek (kaba kuvvet / dallanma yoluyla) doğru çözümü bulabilir.
Bu nedenle sistemin şifre çözme karmaşıklığı q^r ile orantılıdır ve r
küçük tutulduğu sürece pratik kalır.

Sistemin matematiksel bileşenleri şu şekilde özetlenebilir:
    - F_q          : q elemanlı sonlu cisim (q, 2'nin kuvveti olmalı)
    - F_q[x1..xn]   : Açık anahtarın tanımlı olduğu n değişkenli polinom halkası
    - E = F_q^n     : MI çekirdek dönüşümünün çalıştığı genişleme cismi
    - S, T          : Gizli, tersinir afin dönüşümler (S: F_q^n -> F_q^n, T: F_q^n -> F_q^n)
    - F(Y) = Y^e    : MI merkez dönüşümü (e = 1 + q^theta)
    - Z: F_q^n -> F_q^r : Pertürbasyon için lineer izdüşüm matrisi
    - P: F_q^r -> F_q^n : r değişkenli rastgele kuadratik pertürbasyon polinomları

Açık anahtar P_pub, tezin (149. sayfa civarı) verdiği kuruluma göre:
    P_pub(x) = S( F(T(x)) + P(Z(T(x))) )
biçiminde inşa edilir. Burada F(T(x)) MI çekirdeğinin ürettiği "ana" kuadratik
bileşenleri, P(Z(T(x))) ise gürültü bileşenlerini temsil eder.

Bu betik, küçük parametrelerle (GF(4), n=3) çalışan somut bir örnek
üzerinden anahtar üretimi, şifreleme ve şifre çözme (dallanma/tutarlılık
testi) adımlarını ayrıntılı biçimde izlenebilir kılmak amacıyla
hazırlanmıştır.
Referans: Tez, Bölüm 5 — BigField Tabanlı CDPT Kriptosistemleri
"""

from sage.all import *
from itertools import product


class PMI:
    """
    Perturbed Matsumoto-Imai (PMI) kriptosisteminin ayrıntılı 
    SageMath uygulaması.

    Bu sınıf, tezin 5.1.4.1 bölümünde anlatılan PMI kurulumunu adım adım
    gerçekleştirir: gizli anahtarların (S, T, Z, pertürbasyon polinomları)
    üretilmesi, açık anahtarın inşası, şifreleme ve dallanma tabanlı
    şifre çözme.

    Matematiksel Kurulum
    ---------------------
    1. Temel cisim F = GF(q) ve n değişkenli polinom halkası R = F[x1,...,xn].
    2. MI çekirdeğinin çalıştığı genişleme cismi E = F_{q^n}, F üzerinde
       n. dereceden indirgenemez bir polinomla inşa edilir. F_q^n vektör
       uzayı ile E cismi arasında bir vektör uzayı izomorfizması kurulur
       (bkz. extension_to_vector / vector_to_extension metodları).
    3. Pertürbasyon değişkenleri için ayrı, r değişkenli küçük bir
       polinom halkası R_pert = F[z1,...,zr] tanımlanır.
    4. MI üsleri: e = 1 + q^theta olmak üzere, F(Y) = Y^e dönüşümünün
       E* üzerinde bijektif (bire-bir) olabilmesi için gcd(e, q^n - 1) = 1
       koşulunun sağlanması gerekir. Bu koşul sağlandığında d = e^{-1}
       mod (q^n - 1) hesaplanarak tersinir üs (şifre çözmede kullanılacak)
       elde edilir.

    Parametreler
    ------------
    q : int
        Sonlu cismin eleman sayısı. PMI/MI yapısının klasik tanımı
        gereği 2'nin bir kuvveti olması beklenir (karakteristik 2).
    n : int
        Değişken sayısı / genişleme cisminin F_q üzerindeki derecesi.
        gcd(1 + q^theta, q^n - 1) = 1 koşulunu sağlayacak şekilde,
        tercihen n asal seçilmelidir.
    theta : int
        MI merkez dönüşümündeki üs parametresi (e = 1 + q^theta).
    r : int
        Pertürbasyon (gürültü) boyutu. r < n olmalıdır; şifre çözme
        maliyeti q^r ile orantılı olduğundan küçük tutulur.

    Öznitelikler (Anahtar Bileşenleri)
    ------------------------------------
    S_mat, S_vec : Gizli afin dönüşüm S(w) = S_mat * w + S_vec
    T_mat, T_vec : Gizli afin dönüşüm T(x) = T_mat * x + T_vec
    L_mat        : (n x r) boyutlu izdüşüm matrisi (Z dönüşümünün matris karşılığı)
    Perturbation_Polys_z : r değişkenli, n adet kuadratik pertürbasyon polinomu
    Public_Key   : Açık anahtarı oluşturan n değişkenli, n bileşenli
                   kuadratik polinom sistemi

    Notlar
    ------
    Bu sınıf, öğretim ve doğrulama amaçlı ayrıntılı  çıktı
    üretir; performans odaklı bir üretim ortamı implementasyonu değildir.
    """

    def __init__(self, q, n, theta, r):
        """
        PMI (Perturbed Matsumoto-Imai)
        q: Cisim boyutu (2'nin kuvveti olmalı)
        n: Değişken sayısı
        theta: MI parametresi
        r: Pertürbasyon boyutu (r < n)

        Bu kurucu metot, sistemin çalışacağı tüm cebirsel yapıları
        (temel cisim, polinom halkaları, genişleme cismi) kurar ve
        MI üslerinin (e, d) tutarlılığını doğrular. Gizli anahtar
        bileşenleri (S, T, L, pertürbasyon polinomları) burada henüz
        üretilmez; bunlar generate_keys() metodunda oluşturulur.

        Parametreler
        ------------
        q : int
            Sonlu cisim boyutu.
        n : int
            Değişken sayısı / genişleme cisminin derecesi.
        theta : int
            MI merkez dönüşümü üs parametresi.
        r : int
            Pertürbasyon boyutu.

        Olası Hatalar
        -------------------
        ValueError
            gcd(1 + q^theta, q^n - 1) != 1 olduğunda, yani MI merkez
            dönüşümünün E* üzerinde tersinir (bijektif) olmadığı
            durumda oluşur.
        """
        self.q = q
        self.n = n
        self.theta = theta
        self.r = r
        
        # 1. Temel Cisim ve Halka
        # F: q elemanlı sonlu cisim. R: açık anahtarın tanımlı olacağı
        # n değişkenli polinom halkası F[x1,...,xn].
        self.F = GF(q, 'a')
        self.R = PolynomialRing(self.F, n, 'x')
        self.vars = vector(self.R, self.R.gens())
        
        # 2. Genişleme Cismi (MI Çekirdeği)
        # MI merkez dönüşümü F(Y) = Y^e, F_q üzerinde değil, F_{q^n}
        # genişleme cisminde tanımlanır. Bu nedenle önce F_q üzerinde
        # n. dereceden indirgenemez bir polinom (irr_poly) seçilir,
        # ardından E = F_q[X] / (irr_poly) bölüm halkası (bir cisim)
        # inşa edilir. R_E, bu genişleme cismi üzerindeki tek
        # değişkenli polinomları temsil etmek için kullanılır.
        self.R_uni = PolynomialRing(self.F, 'X')
        self.irr_poly = self.R_uni.irreducible_element(n)
        self.E = self.F.extension(self.irr_poly, 'w')
        self.R_E = PolynomialRing(self.E, 'X')
        
        # 3. Pertürbasyon için küçük halka (z1..zr)
        # Pertürbasyon polinomları P(z), n değişkenli asıl halkadan
        # bağımsız olarak, yalnızca r adet yardımcı değişken (z1,...,zr)
        # içeren ayrı bir polinom halkasında tanımlanır. Bu, gürültü
        # bileşenlerinin asıl x değişkenlerinden bağımsız, düşük
        # boyutlu bir uzayda rastgele seçilmesini sağlar.
        self.R_pert = PolynomialRing(self.F, r, 'z')
        self.z_vars = vector(self.R_pert, self.R_pert.gens())
        
        print("\n" + "="*70)
        print(f"PMI SİSTEMİ")
        print(f"Parametreler: GF({q}), n={n}, theta={theta}, r={r}")
        print("="*70)
        
        # MI Üsleri Kontrolü
        # e = 1 + q^theta: MI merkez dönüşümünün üssü. Bu dönüşümün
        # E* (E'nin sıfırdan farklı elemanlarının çarpımsal grubu)
        # üzerinde bir bijeksiyon (bire-bir eşleme) tanımlayabilmesi
        # için gcd(e, q^n - 1) = 1 olması ZORUNLUDUR; aksi halde
        # tersi (d) hesaplanamaz ve şifre çözme mümkün olmaz.
        self.e = 1 + q**theta
        try:
            # d, e'nin (q^n - 1) modülüne göre çarpımsal tersidir.
            # d, şifre çözme aşamasında Y = W^d işlemiyle MI
            # dönüşümünün tersini almak için kullanılır.
            self.d = inverse_mod(self.e, q**n - 1)
            print(f"[KURULUM] MI Üssü e={self.e}")
            print(f"[KURULUM] Ters Üs d={self.d} (GCD=1 Doğrulandı)")
        except:
            raise ValueError(f"HATA: GCD(1+q^theta, q^n-1) != 1. Parametreleri değiştirin (Örn: n={n} yerine n bir asal sayı olsun).")

        # Gizli ve türetilmiş anahtar bileşenleri; generate_keys()
        # çağrılana kadar tanımsız (None) bırakılır.
        self.S_mat = None; self.S_vec = None
        self.T_mat = None; self.T_vec = None
        self.L_mat = None  
        self.Perturbation_Polys_z = [] 
        self.Public_Key = None

    def extension_to_vector(self, element):
        """
        Genişleme cismi E'ye ait bir elemanı, F_q^n vektör uzayındaki
        karşılığına dönüştürür.

        Teorik Gerekçe
        ---------------
        E = F_q[X] / (irr_poly) bölüm halkası, F_q üzerinde n boyutlu
        bir vektör uzayı yapısına da sahiptir (taban: 1, w, w^2, ..., w^(n-1)).
        Bu metot, E cismindeki bir elemanı önce F_q üzerindeki polinom
        temsiline (lift/polynomial) indirger, ardından bu polinomun
        katsayılarını n boyutlu bir vektöre (F_q^n) yerleştirir.
        Bu dönüşüm, MI çekirdeğinin ürettiği sonuçların açık anahtarın
        tanımlı olduğu F_q^n uzayına geri taşınması için gereklidir.

        Parametreler
        ------------
        element : E cismi elemanı (veya E'ye gömülü bir ifade)
            Vektöre dönüştürülecek genişleme cismi elemanı.

        Dönüş Değeri
        ------------
        vector
            F_q üzerinde tanımlı, uzunluğu n olan katsayı vektörü.
        """
        try: poly = element.lift()
        except: 
            try: poly = element.polynomial()
            except: poly = element
        poly = self.R_uni(poly)
        coeffs = poly.list()
        # Katsayı listesi n'den kısa ise (yüksek dereceli terimler
        # sıfır olduğu için), eksik kısımlar 0 ile doldurulur.
        while len(coeffs) < self.n: coeffs.append(self.F(0))
        return vector(self.F, coeffs)

    def vector_to_extension(self, vec):
        """
        F_q^n uzayındaki bir vektörü, genişleme cismi E'ye ait
        karşılığına dönüştürür (extension_to_vector'ın tersi).

        Teorik Gerekçe
        ---------------
        vec = (v0, v1, ..., v_{n-1}) vektörü, E cisminin w tabanına
        göre sum(v_i * w^i) şeklinde yorumlanarak tek bir E elemanına
        eşlenir. Bu işlem, MI merkez dönüşümünün Y^e biçiminde
        genişleme cismi üzerinde uygulanabilmesi için, F_q^n
        vektörlerinin önce E'ye taşınmasını sağlar.

        Parametreler
        ------------
        vec : vector
            F_q üzerinde tanımlı, uzunluğu n olan vektör.

        Dönüş Değeri
        ------------
        E cismi elemanı
            vec vektörünün E cismindeki karşılığı.
        """
        w_gen = self.E.gen()
        res = self.E(0)
        for i in range(self.n):
            res += vec[i] * (w_gen**i)
        return res
    
    def reduce_F_ideal(self, polys):
        """
        Verilen polinom vektörünü, F_q "cisim idealine" (field ideal)
        göre indirger.

        Teorik Gerekçe
        ---------------
        F_q sonlu cisminde, her a elemanı için a^q = a özdeşliği
        geçerlidir (Fermat'nın küçük teoreminin genellemesi). Bu
        nedenle x_i^q - x_i biçimindeki polinomlar, F_q^n üzerindeki
        tüm noktalarda sıfır olur ve bu polinomların ürettiği ideal
        (field ideali, field ideal) ile indirgeme yapmak, polinomların
        F_q^n üzerinde aynı fonksiyonu temsil eden ama daha düşük
        dereceli/normalize edilmiş bir temsilini elde etmeyi sağlar.
        Açık anahtarın kuadratik (derece <= 2) bir sistem olarak
        kalmasını garanti altına almak için bu indirgeme kritik önem
        taşır.

        Parametreler
        ------------
        polys : iterable
            İndirgenecek polinomlar listesi/vektörü.

        Dönüş Değeri
        ------------
        vector
            Cisim idealine göre indirgenmiş polinomlardan oluşan vektör.
        """
        field_eqs = [self.vars[i]**self.q - self.vars[i] for i in range(self.n)]
        FieldIdeal = self.R.ideal(field_eqs)
        return vector(self.R, [FieldIdeal.reduce(p) for p in polys])

    def generate_keys(self):
        """
        PMI gizli anahtarlarını (S, T, L, pertürbasyon polinomları)
        üretir ve bunlardan açık anahtarı (Public_Key) inşa eder.

        Algoritmanın Genel Akışı
        --------------------------
        Tezde (5.1.4.1) tanımlanan kurulum şu adımları izler:
            1. Gizli, tersinir afin dönüşümler S ve T seçilir.
            2. Pertürbasyon bileşenleri için:
               a) L matrisi (n x r): T(x)'in görüntüsünü r boyutlu
                  pertürbasyon uzayına izdüşüren lineer dönüşüm Z'nin
                  matris karşılığıdır.
               b) P(z): r değişkenli, n adet rastgele kuadratik
                  pertürbasyon polinomu.
            3. Açık anahtar:
                   P_pub(x) = S( F(T(x)) + P(L * T(x)) )
               formülüyle hesaplanır. Burada F(T(x)) = MI(y) = y^e
               (y = T(x)) genişleme cisminde hesaplanan "ana" kuadratik
               kısımdır; P(L*y) ise Patarin'in lineerleştirme
               saldırısına karşı koruma sağlayan gürültü kısmıdır.

        Bu yöntemin güvenlik gerekçesi: Standart MI'da x ve y=P(x)
        arasında saldırganın kurabileceği bilineer (lineer) denklemler,
        P(z) gürültüsünün eklenmesiyle artık geçerli olmaz; çünkü
        gürültü bileşeni, MI çekirdeğinin sağladığı katı cebirsel
        yapıyı (bilineer ilişkileri) bozar.

        Dönüş Değeri
        ------------
        vector
            Açık anahtarı oluşturan, F_q üzerinde n bileşenli
            kuadratik polinom sistemi (Public_Key).
        """
        print("\n" + "*"*30 + " ADIM 1: ANAHTAR VE PERTÜRBASYON ÜRETİMİ " + "*"*30)
        
        # 1. S ve T MATRİSLERİ
        # T ve S, sırasıyla girdi (x) ve MI çekirdeğinin çıktısı
        # üzerinde uygulanan gizli, TERSİNİR afin dönüşümlerdir.
        # Tersinirlik şartı, şifre çözme sırasında S^-1 ve T^-1
        # dönüşümlerinin var olabilmesi için zorunludur.
        while True:
            self.T_mat = random_matrix(self.F, self.n, self.n)
            if self.T_mat.is_invertible(): break
        self.T_vec = random_vector(self.F, self.n)
        
        while True:
            self.S_mat = random_matrix(self.F, self.n, self.n)
            if self.S_mat.is_invertible(): break
        self.S_vec = random_vector(self.F, self.n)
        
        print(f"\n[1.1] Gizli Afin Dönüşümler:")
        print(f"   T Matrisi ({self.n}x{self.n}):\n{self.T_mat}")
        print(f"   c_T: {self.T_vec}")
        print(f"   S Matrisi ({self.n}x{self.n}):\n{self.S_mat}")
        print(f"   c_S: {self.S_vec}")

        # 2. PERTÜRBASYON (BOZMA) BİLEŞENLERİ
        print(f"\n[1.2] Pertürbasyon (Gürültü) Bileşenleri:")
        
        # L Matrisi (n x r)
        # Z: F_q^n -> F_q^r lineer izdüşüm dönüşümünün matris
        # karşılığıdır. y = T(x) vektörü, y * L_mat işlemiyle
        # r boyutlu pertürbasyon uzayına indirgenir.
        self.L_mat = random_matrix(self.F, self.n, self.r)
        print(f"   L Matrisi (İzdüşüm - {self.n}x{self.r}):\n{self.L_mat}")
        
        # Pertürbasyon Polinomları P(z)
        # Her biri r değişkenli (z1,...,zr), toplam n adet rastgele
        # kuadratik polinom seçilir. Bu polinomlar, açık anahtara
        # gürültü olarak eklenecek P(z) dönüşümünü oluşturur.
        self.Perturbation_Polys_z = []
        print(f"   Pertürbasyon Polinomları P(z) (r={self.r} değişkenli):")
        
        for k in range(self.n):
            poly = self.R_pert.random_element(degree=2)
            self.Perturbation_Polys_z.append(poly)
            print(f"      p_{k}(z) = {poly}")
        
        # 3. AÇIK ANAHTAR İNŞASI
        print(f"\n[1.3] Açık Anahtar P_pub(x) Hesaplanıyor...")
        print(f"      Formül: P_pub = S( MI(y) + P(L*y) )  nerede y = T(x)")
        
        # A) T(x)
        # y = T(x): x değişkenleri üzerinde sembolik olarak
        # hesaplanan afin dönüşüm sonucu.
        y_vec = self.T_mat * self.vars + self.T_vec
        print(f"   -> y = T(x) sembolik olarak hesaplandı.")
        
        # B) MI(y) = y^e
        # MI çekirdeğinin sembolik olarak (x değişkenleri cinsinden)
        # hesaplanabilmesi için, y_vec bileşenleri genişleme cismi E
        # üzerindeki çok değişkenli bir polinom halkasına taşınır.
        R_E_multi = PolynomialRing(self.E, self.n, 'x')
        y_vec_E = vector(R_E_multi, [R_E_multi(p) for p in y_vec])
        
        w_gen = self.E.gen()
        # Y_sym: y_vec'in E tabanına (1, w, w^2, ...) göre sembolik
        # toplamı; yani y vektörünün E cismindeki sembolik temsilidir.
        Y_sym = sum(y_vec_E[i] * (w_gen**i) for i in range(self.n))
        
        # Z = Y^e : MI merkez dönüşümünün sembolik olarak uygulanması.
        Z_sym = Y_sym ** self.e
        
        # İndirgeme ve Vektöre Dönüş
        # Genişleme cismi üzerinde hesaplanan Z_sym, F_q'nun cisim
        # özdeşliği (x_i^q = x_i) kullanılarak indirgenir; böylece
        # sonucun kuadratik dereceyi aşmaması sağlanır.
        print(f"   -> MI(y) Genişleme cisminde hesaplanıyor...")
        vars_E = R_E_multi.gens()
        field_eqs_E = [vars_E[i]**self.q - vars_E[i] for i in range(self.n)]
        Ideal_E = R_E_multi.ideal(field_eqs_E)
        Z_reduced = Ideal_E.reduce(Z_sym)
        
        # Z_reduced, E cismi katsayılı bir polinomdur. Bu katsayılar
        # tek tek F_q^n'e (extension_to_vector ile) geri açılarak,
        # MI(y)'nin F_q üzerindeki n bileşenli polinom vektör temsili
        # (mi_polys) elde edilir.
        mi_polys = [self.R(0) for _ in range(self.n)]
        for exps, coeff in Z_reduced.dict().items():
            coeff_vec = self.extension_to_vector(coeff)
            monomial = self.R.monomial(*exps)
            for k in range(self.n):
                mi_polys[k] += coeff_vec[k] * monomial
        MI_vec = vector(self.R, mi_polys)
        
        # --- YENİ EKLENEN KISIM: MI(y) YAZDIRMA ---
        print(f"\n   [DETAY] MI(y) Sembolik Vektörü (İlk 3 bileşen):")
        for i, p in enumerate(MI_vec[:3]):
            print(f"      MI_{i} = {p}")
        # ------------------------------------------

        # C) P(L*y) - Gürültü Kısmı
        # Pertürbasyon polinomları, z değişkenleri yerine y*L_mat
        # ifadesi konularak x cinsinden ifade edilir. Bu, gürültü
        # bileşeninin de açık anahtarın x değişkenleri üzerinden
        # tanımlı bir polinom sistemine dönüştürülmesini sağlar.
        print(f"\n   -> Pertürbasyon P(L*y) hesaplanıyor...")
        z_in_y = y_vec * self.L_mat
        sub_dict_z = {self.z_vars[i]: z_in_y[i] for i in range(self.r)}
        
        Pert_vec_y = vector(self.R, [p.subs(sub_dict_z) for p in self.Perturbation_Polys_z])
        
        # --- YENİ EKLENEN KISIM: P(L*y) YAZDIRMA ---
        print(f"   [DETAY] P(L*y) Sembolik Vektörü (İlk 3 bileşen):")
        for i, p in enumerate(Pert_vec_y[:3]):
            print(f"      P_pert_{i} = {p}")
        # -------------------------------------------

        # D) Toplam ve S
        # Ana kısım (MI(y)) ile gürültü kısmı (P(L*y)) toplanarak
        # MI çekirdeğinin "bozulmuş" (perturbed) hali elde edilir.
        # Ardından bu toplam, gizli S afin dönüşümünden geçirilerek
        # ham açık anahtar hesaplanır ve cisim idealine göre indirgenir.
        Combined = MI_vec + Pert_vec_y
        
        raw_pk = self.S_mat * Combined + self.S_vec
        self.Public_Key = self.reduce_F_ideal(raw_pk)
        
        # Derece Kontrolü
        # PMI'nin bir Multivariate Quadratic (MQ) sistem olarak
        # kabul edilebilmesi için açık anahtardaki her polinomun
        # derecesinin en fazla 2 olması gerekir.
        max_deg = max(p.degree() for p in self.Public_Key)
        print(f"\n   [SONUÇ] Açık Anahtar Hazır. Maksimum Derece: {max_deg}")
        if max_deg <= 2:
            print("   -> DOĞRULAMA: Sistem geçerli bir Multivariate Quadratic (MQ) sistemdir.")
        else:
            print("   -> HATA: Derece 2'den büyük! Parametreleri kontrol edin.")

        for i, p in enumerate(self.Public_Key):
            print(f"      Pub_{i} = {p}")
        
        return self.Public_Key

    def encrypt(self, message_vec, debug=True):
        """
        Verilen düz metin vektörünü, PMI açık anahtarını kullanarak
        şifreler.

        Şifreleme adımları, generate_keys() içindeki sembolik açık
        anahtar inşasının, somut bir mesaj vektörü üzerinde sayısal
        olarak (F_q elemanlarıyla) tekrar edilmesinden ibarettir:

            1. y = T(x)               (gizli afin dönüşüm)
            2. z = y * L               (pertürbasyon uzayına izdüşüm)
            3. p_noise = P(z)          (gürültü bileşeni)
            4. mi = MI(y) = y^e        (MI çekirdeği, E cisminde)
            5. c = S(mi + p_noise)     (gizli afin dönüşüm ile şifreleme)

        Parametreler
        ------------
        message_vec : vector
            Şifrelenecek düz metin; F_q üzerinde uzunluğu n olan vektör.
        debug : bool, opsiyonel (varsayılan True)
            True ise ara adımların (y, z, gürültü, MI(y), toplam, çıktı)
            ayrıntılı çıktısı ekrana yazdırılır.

        Dönüş Değeri
        ------------
        vector
            Şifreli metin; F_q üzerinde uzunluğu n olan vektör.
        """
        # 1. T Dönüşümü: y = T(x)
        y_vec = self.T_mat * message_vec + self.T_vec
        
        # 2. L İzdüşümü
        # z = y * L: y vektörünün pertürbasyon uzayına (r boyutlu)
        # izdüşümü. Bu z değeri, hangi pertürbasyon polinom değerinin
        # gürültü olarak ekleneceğini belirler.
        z_vec = y_vec * self.L_mat
        
        # 3. P(z) yani P(L*y) Değerinin Hesaplanması
        sub_dict = {self.z_vars[i]: z_vec[i] for i in range(self.r)}
        p_noise_vec = vector(self.F, [p.subs(sub_dict) for p in self.Perturbation_Polys_z])
        
        # 4. MI Çekirdeği: MI(y) = y^e
        # y vektörü önce E cismine taşınır (vector_to_extension),
        # ardından e. kuvveti alınır ve sonuç tekrar F_q^n vektörüne
        # (extension_to_vector) dönüştürülür.
        Y_elem = self.vector_to_extension(y_vec)
        Z_elem = Y_elem ** self.e
        mi_vec = self.extension_to_vector(Z_elem)
        
        # 5. Toplama ve S Dönüşümü
        # MI çekirdeğinin çıktısı ile gürültü bileşeni toplanır ve
        # gizli S afin dönüşümü uygulanarak şifreli metin elde edilir.
        combined = mi_vec + p_noise_vec
        ciphertext = self.S_mat * combined + self.S_vec
        
        if debug:
            print(f"\n   [ENCRYPT İŞLEM ADIMLARI]")
            print(f"   1. Girdi (x)         : {message_vec}")
            print(f"   2. y = T(x)          : {y_vec}")
            print(f"   3. z = y * L         : {z_vec}")
            print(f"   --------------------------------------------------")
            print(f"   4. P(L*y) (Gürültü)  : {p_noise_vec}")
            print(f"   5. MI(y)  (Ana Kısım): {mi_vec}")
            print(f"   --------------------------------------------------")
            print(f"   6. Toplam (MI + P)   : {combined}")
            print(f"   7. Çıktı S(Toplam)   : {ciphertext}")
            
        return ciphertext

    def decrypt(self, ciphertext):
        """
        Verilen şifreli metni, gizli anahtar bileşenleri (S, T, Z,
        pertürbasyon polinomları, d) yardımıyla çözer.

        Algoritmanın Teorik Temeli: Dallanma (Branching) ve
        Tutarlılık Testi
        -------------------------------------------------------------
        Standart MI'da şifre çözme, tek bir üs alma işlemiyle
        (Y = W^d) doğrudan gerçekleştirilebilir. PMI'da ise açık
        anahtar MI(y) ile birlikte bilinmeyen bir P(z) gürültü terimi
        de içerdiğinden, alıcı hedef vektör (y_target = MI(y) + P(z))
        içindeki P(z) katkısını bilmeden doğrudan MI'nin tersini alamaz.

        Bu belirsizliğin üstesinden gelmek için "dallanma" (branching)
        stratejisi izlenir: pertürbasyon değeri z, r boyutlu bir vektör
        olduğundan olası z değerlerinin sayısı yalnızca q^r kadardır
        (r küçük tutulduğundan bu sayı pratikte yönetilebilir düzeydedir).
        Her z adayı için:

            1. O z adayına karşılık gelen gürültü P(z) hesaplanır.
            2. Hedeften bu gürültü çıkarılarak "saf" MI çıktısı adayı
               (mi_pure_vec = y_target - P(z)) elde edilir.
            3. Bu saf MI çıktısı E cismine taşınıp d. kuvveti alınarak
               (Y = W^d) bir y adayı hesaplanır (MI'nin tersi).
            4. TUTARLILIK TESTİ: Eğer tahmin doğruysa, bulunan y adayı
               ile y*L çarpımı, başlangıçta varsayılan z değerine eşit
               olmalıdır (z_check == z_vec). Bu kontrol, doğru z
               adayını yanlış olanlardan ayırt etmeyi sağlar.
            5. Tutarlılık testini geçen adaylar için T^-1 uygulanarak
               olası düz metin adayları elde edilir.

        Bu yöntem, tezin 5.1.4.1 bölümünde açıklanan "her olası z
        değerini deneyerek Z(y) = z eşitliğini sağlayan çözümü bulma"
        yaklaşımının doğrudan bir uygulamasıdır.

        Parametreler
        ------------
        ciphertext : vector
            Çözülecek şifreli metin; F_q üzerinde uzunluğu n olan vektör.

        Dönüş Değeri
        ------------
        list
            Tutarlılık testini geçen tüm düz metin adaylarının listesi.
            İdeal koşullarda bu liste tek bir doğru çözüm içerir; ancak
            teoride birden fazla aday da ortaya çıkabilir (bu durumun
            PMI+ gibi varyantlarla nasıl azaltıldığı tezde ayrıca ele
            alınmıştır).
        """
        print("\n" + "*"*30 + " ADIM 2: DEŞİFRELEME (PMI) " + "*"*30)
        print(f"   Şifreli Metin (c): {ciphertext}")
        
        # 1. S^-1
        # y_target = MI(y) + P(z)
        # Gizli S dönüşümünün tersi uygulanarak, MI çekirdeği ile
        # gürültünün toplamından oluşan hedef vektör elde edilir.
        y_target = self.S_mat.inverse() * (ciphertext - self.S_vec)
        print(f"\n[2.1] S^-1 Uygulandı. Hedef Vektör (y_target): {y_target}")
        
        # 2. BRANCHING (Dallanma)
        print(f"\n[2.2] Pertürbasyon (z) Tahmini ve Tutarlılık Testi")
        print(f"      Mantık: z vektörünü ({self.r} boyutlu) tahmin et, gürültüyü çıkar, MI tersini al.")
        print(f"      Toplam Aday Sayısı: {self.q}^{self.r} = {self.q**self.r}")
        
        candidates = []
        possible_vals = list(self.F)
        
        # F_q^r uzayındaki TÜM olası z vektörleri (q^r adet) sırayla
        # denenir; bu, PMI'nin dallanma tabanlı şifre çözme stratejisinin
        # doğrudan uygulamasıdır.
        for z_val in product(possible_vals, repeat=self.r):
            z_vec = vector(self.F, z_val)
            
            # A) Bu z tahmini için Gürültü (Noise) vektörünü hesapla: P(z)
            sub_dict = {self.z_vars[i]: z_vec[i] for i in range(self.r)}
            noise_vec = vector(self.F, [p.subs(sub_dict) for p in self.Perturbation_Polys_z])
            # --- EKLENEN SATIR: Her z ve P(z) ikilisini yazdır ---
            print(f"      -> Aday z: {z_vec}  =>  Gürültü P(z): {noise_vec}")
            # -----------------------------------------------------
            # B) Saf MI değeri: MI(y) = Hedef - Noise
            # Tahmin edilen gürültü, hedeften çıkarılarak MI çekirdeğinin
            # (gürültüsüz) çıktısına ait bir aday elde edilir.
            mi_pure_vec = y_target - noise_vec
            
            # C) MI Tersini Al (y = MI^-1)
            # Saf MI çıktısı E cismine taşınır ve d. kuvveti alınarak
            # (e'nin çarpımsal tersi olduğundan) MI dönüşümünün tersi
            # hesaplanır: eğer W = Y^e ise Y = W^d.
            W = self.vector_to_extension(mi_pure_vec)
            Y = W ** self.d # Y = W^d
            y_candidate = self.extension_to_vector(Y)
            
            # D) TUTARLILIK TESTİ (CONSISTENCY CHECK)
            # Eğer bu y doğruysa, L*y işlemi tahmin ettiğimiz z'yi vermeli!
            # z_check = y * L
            # Bu kontrol, yanlış z tahminlerinin ürettiği sahte y
            # adaylarını elemek için kullanılan temel doğrulama
            # mekanizmasıdır.
            z_check = y_candidate * self.L_mat
            
            if z_check == z_vec:
                print(f"\n   >>> [EŞLEŞME BULUNDU!]")
                print(f"       Tahmin Edilen z: {z_vec}")
                print(f"       Hesaplanan Gürültü: {noise_vec}")
                print(f"       Bulunan y (MI Tersinden): {y_candidate}")
                print(f"       Tutarlılık Kontrolü (y*L == z): {z_check == z_vec} (BAŞARILI)")
                
                # E) T^-1
                # Tutarlılık testini geçen y adayına gizli T dönüşümünün
                # tersi uygulanarak orijinal düz metin adayı elde edilir.
                plaintext = self.T_mat.inverse() * (y_candidate - self.T_vec)
                print(f"       T^-1 Uygulandı. Mesaj Adayı (x): {plaintext}")
                candidates.append(plaintext)
            
        return candidates


    def public_map(self, message_vec, debug=True):
        """
        Açık anahtar dönüşümünün (P_pub) bir kısaltması; encrypt()
        metoduna eşdeğerdir.
        """
        return self.encrypt(message_vec, debug=debug)

    def invert_with_secret_key(self, ciphertext):
        """
        Gizli anahtar kullanılarak açık anahtar dönüşümünün tersinin
        alınması; decrypt() metoduna eşdeğerdir.
        """
        return self.decrypt(ciphertext)

In [4]:
# ============================================================================
# SENARYO
# ============================================================================
# Bu bölüm, yukarıda tanımlanan PMI sınıfının küçük, somut bir
# parametre seti (GF(4), n=3) üzerinde uçtan uca (anahtar üretimi ->
# şifreleme -> şifre çözme) test edilmesini sağlar. Amaç, tezin
# 5.1.4.1 bölümünde teorik olarak sunulan yapının doğru işlediğini
# doğrulamak ve ara adımları somut sayısal değerlerle izlenebilir
# kılmaktır.
print("\n\n" + "*"*60)
print(">>> PMI SİMÜLASYONU (GF(4)) <<<")
print("*"*60)

# Parametreler
# q=4 (Characteristic 2)
# n=3 (Değişken sayısı - GCD sorunu çıkmasın diye 3 seçildi)
# theta=1 
# r=1 (Küçük pertürbasyon, 4 deneme yapar)

my_q = 4
my_n = 3
my_theta = 2
my_r = 1

pmi = PMI(q=my_q, n=my_n, theta=my_theta, r=my_r)
pmi.generate_keys()

# Test Mesajı
# F_q^n uzayından rastgele seçilen bir düz metin vektörü ile
# şifreleme/şifre çözme döngüsünün doğruluğu sınanır.
msg = random_vector(pmi.F, my_n)
print(f"\n[TEST] Orijinal Mesaj: {msg}")

cipher = pmi.encrypt(msg)
print(f"[TEST] Şifreli Metin: {cipher}")

# Şifre çözme, dallanma stratejisiyle olası tüm z değerlerini
# tarayarak tutarlılık testini geçen düz metin adaylarını döndürür.
adaylar = pmi.decrypt(cipher)

print("\n" + "="*60)
if msg in adaylar:
    print(f"SONUÇ: BAŞARILI. Orijinal mesaj bulundu.")
else:
    print(f"SONUÇ: HATA.")
print("="*60)



************************************************************
>>> PMI SİMÜLASYONU (GF(4)) <<<
************************************************************

PMI SİSTEMİ
Parametreler: GF(4), n=3, theta=2, r=1
[KURULUM] MI Üssü e=17
[KURULUM] Ters Üs d=26 (GCD=1 Doğrulandı)

****************************** ADIM 1: ANAHTAR VE PERTÜRBASYON ÜRETİMİ ******************************

[1.1] Gizli Afin Dönüşümler:
   T Matrisi (3x3):
[    a a + 1     0]
[    0     1     1]
[a + 1     1     1]
   c_T: (1, 0, a + 1)
   S Matrisi (3x3):
[    1     1 a + 1]
[a + 1     a a + 1]
[a + 1     a     1]
   c_S: (1, a, a + 1)

[1.2] Pertürbasyon (Gürültü) Bileşenleri:
   L Matrisi (İzdüşüm - 3x1):
[a + 1]
[    0]
[    a]
   Pertürbasyon Polinomları P(z) (r=1 değişkenli):
      p_0(z) = z^2 + (a)*z + 1
      p_1(z) = (a + 1)*z^2 + (a)*z + (a)
      p_2(z) = (a)*z^2 + (a + 1)

[1.3] Açık Anahtar P_pub(x) Hesaplanıyor...
      Formül: P_pub = S( MI(y) + P(L*y) )  nerede y = T(x)
   -> y = T(x) sembolik olarak he